## 5. Evaluación en todo el conjunto de datos (Opcional)

Para evaluar el error en todo el conjunto completo (10.000 imágenes de test) usamos la misma función `fgsm` definida en el apartado 3. Como el tensor con 10.000 imágenes es grande, la función `fgsm` consumirá más memoria, pero al tratarse de las imágenes de MNIST (28x28) habitualmente se puede procesar en una sola pasada.

El objetivo aquí es obtener una métrica real de cómo se degradan ambos modelos (Red Neuronal y SVM) cuando se enfrentan a un conjunto de test formado por imágenes adversarias.

In [1]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist
from tensorflow.keras import layers, models
from sklearn.svm import SVC

# fijamos la semilla de Python, NumPy y TensorFlow 
tf.keras.utils.set_random_seed(42)
# fuerza operaciones deterministas 
tf.config.experimental.enable_op_determinism()

I0000 00:00:1794763260.734856   18236 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794763260.861648   18236 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794763262.321211   18236 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [ ]:
# -----------------------------------------------------------------------------
# APARTADO 5: Generar adversarias para test completo y evaluar error
# -----------------------------------------------------------------------------

print("Generando 10.000 imágenes adversarias para el conjunto de test...")
# Atacamos el conjunto de test completo con la red neuronal original
x_test_adv_full = fgsm(red_neuronas, x_test, y_test, EPSILON)

# 1. Error de la Red Neuronal sobre el conjunto adversario
perdida_rn_adv, acierto_rn_adv = red_neuronas.evaluate(x_test_adv_full, y_test, verbose=0)
error_rn_adv = 1.0 - acierto_rn_adv
print(f"Acierto Red Neuronal (test adversario): {acierto_rn_adv:.4f} -> Error: {error_rn_adv:.4f}")

# 2. Error de la SVM sobre el conjunto adversario
# Aplanamos las imágenes para pasarlas por la SVM
x_test_adv_full_svm = x_test_adv_full.reshape(len(x_test_adv_full), -1)
acierto_svm_adv = svm.score(x_test_adv_full_svm, y_test)
error_svm_adv = 1.0 - acierto_svm_adv
print(f"Acierto SVM (test adversario): {acierto_svm_adv:.4f} -> Error: {error_svm_adv:.4f}")

## 6. Entrenamiento adversario (Opcional)

Para entrenar con 120.000 imágenes, primero necesitamos generar las 60.000 imágenes adversarias del conjunto de entrenamiento. Al hacer pasar 60.000 imágenes de golpe por `tf.GradientTape`, es muy probable que ocurra un error de memoria. Por ello, se crea la función auxiliar `fgsm_batch`, que aplica el ataque procesando las imágenes en lotes.

Una vez generadas:
1. Concatenamos los datos originales con los adversarios para obtener un conjunto de entrenamiento de 120.000 imágenes y un test de 20.000.
2. Barajamos el conjunto de entrenamiento para que la red no lea primero todas las originales y luego todas las adversarias.
3. Entrenamos desde cero una **nueva red neuronal** y una **nueva SVM** con este conjunto combinado para hacerlas más robustas.


In [ ]:
# -----------------------------------------------------------------------------
# APARTADO 6: Entrenamiento adversario con 120.000 imágenes
# -----------------------------------------------------------------------------

def fgsm_batch(modelo, x, y, epsilon, batch_size=2000):
    """Aplica FGSM procesando las imágenes en pequeños lotes para no saturar la memoria."""
    x_adv = np.empty_like(x)
    for i in range(0, len(x), batch_size):
        x_batch = x[i:i+batch_size]
        y_batch = y[i:i+batch_size]
        x_adv[i:i+batch_size] = fgsm(modelo, x_batch, y_batch, epsilon)
    return x_adv

print("Generando 60.000 imágenes adversarias de entrenamiento...")
x_train_adv_full = fgsm_batch(red_neuronas, x_train, y_train, EPSILON)

# Concatenamos los datos para obtener los conjuntos conjuntos (120k train, 20k test)
x_train_120k = np.concatenate([x_train, x_train_adv_full])
y_train_120k = np.concatenate([y_train, y_train])

x_test_20k = np.concatenate([x_test, x_test_adv_full])
y_test_20k = np.concatenate([y_test, y_test])

# Barajamos aleatoriamente el nuevo conjunto de train para que la red no lea 
# primero todas las originales y luego todas las adversarias (fijando semilla)
rng = np.random.default_rng(42)
indices_120k = rng.permutation(len(x_train_120k))
x_train_120k = x_train_120k[indices_120k]
y_train_120k = y_train_120k[indices_120k]

# Creamos las vistas aplanadas para entrenar la SVM robusta
x_train_120k_svm = x_train_120k.reshape(len(x_train_120k), -1)
x_test_20k_svm = x_test_20k.reshape(len(x_test_20k), -1)

# Entrenamos una NUEVA Red Neuronal idéntica a la del apartado 1
red_neuronas_adv = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(8, (5, 5), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(24, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax"),
])

red_neuronas_adv.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.1),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

print("Entrenando Red Neuronal robusta (120.000 imágenes)...")
red_neuronas_adv.fit(x_train_120k, y_train_120k, epochs=15, batch_size=64, validation_split=0.1)

# Evaluamos la Red Neuronal robusta en el test conjunto de 20.000
perdida_rn_20k, acierto_rn_20k = red_neuronas_adv.evaluate(x_test_20k, y_test_20k, verbose=0)
print(f"Error de la Red Neuronal robusta en test conjunto (20k): {1.0 - acierto_rn_20k:.4f}")

# Entrenamos la NUEVA SVM robusta
# AVISO: Entrenar una SVM es de complejidad casi O(N^2) o O(N^3). Con 120.000 datos
# tardará significativamente más que en el apartado 2 (puede llevar desde varios minutos a un par de horas).
print("Entrenando SVM robusta (120.000 imágenes)...")
svm_adv = SVC(kernel="rbf", C=10, gamma=0.01)
svm_adv.fit(x_train_120k_svm, y_train_120k)

# Evaluamos la SVM robusta en el test conjunto de 20.000
acierto_svm_20k = svm_adv.score(x_test_20k_svm, y_test_20k)
print(f"Error de la SVM robusta en test conjunto (20k): {1.0 - acierto_svm_20k:.4f}")

## 7. Imágenes “doblemente adversarias” (Opcional)

En este apartado tomamos las 60.000 imágenes de entrenamiento que ya tenían el ruido $\epsilon$ inyectado respecto a la red original, y las **volvemos a atacar**. 

Para medir verdaderamente la robustez del nuevo modelo entrenado adversariamente, calculamos los gradientes respecto a esta *nueva* red neuronal (`red_neuronas_adv`) y sumamos el nuevo error a la imagen que ya estaba previamente modificada. Finalmente, comprobamos qué rendimiento obtiene el modelo robusto sobre estas imágenes.

In [ ]:
# -----------------------------------------------------------------------------
# APARTADO 7: Ataque doblemente adversario
# -----------------------------------------------------------------------------

print("Generando imágenes doblemente adversarias (atacando 60.000 de train)...")
# Atacamos las imágenes que ya eran adversarias (x_train_adv_full).
# Utilizamos la nueva red_neuronas_adv para generar el gradiente de este segundo ataque.
x_train_adv_doble = fgsm_batch(red_neuronas_adv, x_train_adv_full, y_train, EPSILON)

# Calculamos el rendimiento del modelo robusto sobre estas 60.000 doblemente adversarias
perdida_doble, acierto_doble = red_neuronas_adv.evaluate(x_train_adv_doble, y_train, verbose=0)
print(f"Rendimiento de la Red Neuronal robusta en Train doblemente adversario (60k): {acierto_doble:.4f}")

# Como añadido adicional, hacemos lo mismo para el conjunto de test para ver la generalización (10k)
x_test_adv_doble = fgsm_batch(red_neuronas_adv, x_test_adv_full, y_test, EPSILON)
perdida_doble_test, acierto_doble_test = red_neuronas_adv.evaluate(x_test_adv_doble, y_test, verbose=0)
print(f"Rendimiento de la Red Neuronal robusta en Test doblemente adversario (10k): {acierto_doble_test:.4f}")